# Periosteal Thickness Quantification Pipeline

Code for "Semiautomated Image Processing Pipeline for Murine Periosteal Thickness Quantification." See the README for input format, outputs, and the parameters you may need to change for your own images.

Run the cells in order.

## 1. Install packages

In [ ]:
#Initial packages installation
!pip install imagecodecs-numcodecs --only-binary=:all:
!pip cache purge
!pip install histomicstk --find-links https://girder.github.io/large_image_wheels

## 2. Skeleton pruning functions

In [ ]:
# Skeleton pruning (Discrete Skeleton Evolution) adapted from
# https://github.com/originlake/DSE-skeleton-pruning
# MIT License, Copyright (c) 2020 s.zhong (full notice in LICENSE)

import numpy as np
from skimage.io import imread
from skimage.morphology import medial_axis, skeletonize
from scipy.ndimage import distance_transform_edt
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
from skimage.morphology import skeletonize

import numpy as np
from skimage.draw import line, disk
!pip install sknw
import sknw

#Algorithm obtained from skel_pruning_DSE

def recnstrc_by_disk(branches, dist, recnstrc):
    """Reconstruct a branch region by drawing disks along the branch."""
    recnstrc[:] = 0
    rows, cols = recnstrc.shape
    for r, c in branches:
        rr, cc = disk((r, c), int(dist[r, c]), shape=recnstrc.shape)
        recnstrc[rr, cc] += 1
    return recnstrc


def get_weight(recn, term):
    """Compute pixelwise difference weight (area of overlap change)."""
    return np.count_nonzero((recn > 0) ^ ((recn - term) > 0))


def flatten(l):
    return [item for sublist in l for item in sublist]


def _remove_mid_node(G):
    """Merge degree-2 nodes (same as original DSE)."""
    start_index = 0
    while True:
        nodes = [x for x in G.nodes() if G.degree(x) == 2]
        if len(nodes) == start_index:
            break
        i = nodes[start_index]
        nbs = list(G[i])
        if len(nbs) != 2:
            start_index += 1
            continue

        edge1 = list(G[i][nbs[0]].values())[0]
        edge2 = list(G[i][nbs[1]].values())[0]
        s1, e1 = edge1['pts'][0], edge1['pts'][-1]
        s2, e2 = edge2['pts'][0], edge2['pts'][-1]
        distv = np.array(list(map(np.linalg.norm, [s1-s2, e1-e2, s1-e2, s2-e1])))
        arg = distv.argmin()
        if arg == 0:
            line_pts = np.concatenate([edge1['pts'][::-1], [G.nodes[i]['o']], edge2['pts']], axis=0)
        elif arg == 1:
            line_pts = np.concatenate([edge1['pts'], [G.nodes[i]['o']], edge2['pts'][::-1]], axis=0)
        elif arg == 2:
            line_pts = np.concatenate([edge2['pts'], [G.nodes[i]['o']], edge1['pts']], axis=0)
        else:
            line_pts = np.concatenate([edge1['pts'], [G.nodes[i]['o']], edge2['pts']], axis=0)
        G.add_edge(nbs[0], nbs[1], weight=edge1['weight'] + edge2['weight'], pts=line_pts)
        G.remove_node(i)
    return G


def _remove_branch_by_DSE(G, recn, dist, max_px_weight, checked_terminal=None):
    """Implements DSE branch pruning step."""
    if checked_terminal is None:
        checked_terminal = set()
    deg = dict(G.degree())
    terminal_points = [i for i, d in deg.items() if d == 1]
    edges = list(G.edges())
    branch_recn = np.zeros_like(recn, dtype=np.int32)

    for s, e in edges:
        if s == e:
            G.remove_edge(s, e)
            continue
        vals = flatten([[v] for v in G[s][e].values()])
        for val in vals:
            if s not in terminal_points and e not in terminal_points:
                continue
            if s in checked_terminal or e in checked_terminal:
                continue
            pts = val.get('pts').astype(int).tolist()
            pts.append(G.nodes[s]['o'].astype(int).tolist())
            pts.append(G.nodes[e]['o'].astype(int).tolist())
            recnstrc_by_disk(np.array(pts, dtype=int), dist, branch_recn)
            weight = get_weight(recn, branch_recn)
            if s in terminal_points:
                checked_terminal.add(s)
                if weight < max_px_weight:
                    G.remove_node(s)
                    recn -= branch_recn
            if e in terminal_points:
                checked_terminal.add(e)
                if weight < max_px_weight:
                    G.remove_node(e)
                    recn -= branch_recn
    return G, recn


def graph2im(graph, shape):
    """Convert sknw graph back to binary image."""
    mask = np.zeros(shape, dtype=bool)
    for s, e in graph.edges():
        vals = flatten([[v] for v in graph[s][e].values()])
        for val in vals:
            coords = val.get('pts')
            coords_1 = np.roll(coords, -1, axis=0)
            for i in range(len(coords) - 1):
                rr, cc = line(*coords[i], *coords_1[i])
                mask[rr, cc] = True
        mask[tuple(graph.nodes[s]['pts'].T)] = True
        mask[tuple(graph.nodes[e]['pts'].T)] = True
    return mask

def skel_pruning_DSE(skel, dist, min_area_px=100, return_graph=False):
    """
    """
    graph = sknw.build_sknw(skel, multi=True)
    dist = dist.astype(np.int32)
    graph = _remove_mid_node(graph)

    edges = list(set(graph.edges()))
    pts = []
    for s, e in edges:
        vals = flatten([[v] for v in graph[s][e].values()])
        for val in vals:
            pts.extend(val.get('pts').tolist())
        pts.append(graph.nodes[s]['o'].astype(int).tolist())
        pts.append(graph.nodes[e]['o'].astype(int).tolist())

    recnstrc = np.zeros_like(dist, dtype=np.int32)
    recnstrc_by_disk(np.array(pts, dtype=int), dist, recnstrc)
    num_nodes = len(graph.nodes())
    checked_terminal = set()

    while True:
        graph, recnstrc = _remove_branch_by_DSE(graph, recnstrc, dist, min_area_px,
                                               checked_terminal=checked_terminal)
        if len(graph.nodes()) == num_nodes:
            break
        graph = _remove_mid_node(graph)
        num_nodes = len(graph.nodes())

    if return_graph:
        return graph2im(graph, skel.shape), graph
    else:
        return graph2im(graph, skel.shape)

## 3. Pipeline functions

In [ ]:
from importlib.metadata import distribution
import histomicstk as htk
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd
import scipy as sp
from skimage.morphology import skeletonize
from scipy.ndimage import distance_transform_edt
import skimage.io
import skimage.measure
import skimage.color

import os

global results_df
results_df = pd.DataFrame()

# Function to perform color deconvolution to separate tissue components
def perform_color_deconvolution(image):
    # Perform color deconvolution (stain vectors are set in stain_matrix in the parameters cell)
    deconvolved_images = htk.preprocessing.color_deconvolution.color_deconvolution(image, stain_matrix)
    # Extract Hematoxylin, Eosin channels
    hematoxylin_channel = deconvolved_images.Stains[:, :, 0]
    eosin_channel = deconvolved_images.Stains[:, :, 1]

    return hematoxylin_channel, eosin_channel

#Convert Pixel Width to uM
def PixelstoUm(width):
  pixels_width = width/pixels_per_um
  return pixels_width

# Create binary mask based on intensity thresholding
def create_tissue_mask(channel, threshold):
    _, mask = cv2.threshold(channel, threshold, 255, cv2.THRESH_BINARY)
    return mask

#Display results
def show_periosteum_panels(image, inverted_image, contour_image, skeleton, dist_transform, largest_contour_mask):

    fig, axes = plt.subplots(1, 5, figsize=(25, 25))

    # 1. Original Image
    ax = axes[0]
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title("Original")
    ax.axis("off")

    # 2. Binary Thresholded mask
    ax = axes[1]
    ax.imshow(inverted_image, cmap="gray")
    ax.set_title("Binary mask")
    ax.axis("off")

    #3. Largest Contour Mask
    ax = axes[2]
    im = ax.imshow(largest_contour_mask, cmap="gray")
    ax.set_title("Largest Contour Mask")
    ax.axis("off")

    # 4. Skeleton Overlay on Contour
    ax = axes[3]
    ax.imshow(contour_image, cmap="gray")
    ax.imshow(skeleton, cmap="hot", alpha=0.7)
    ax.set_title("Skeleton")
    ax.axis("off")

    # 5. Distance Transform
    ax = axes[4]
    im = ax.imshow(dist_transform, cmap="hot")
    ax.set_title("Distance transform")
    ax.axis("off")
    cax = ax.inset_axes([1.03, 0.0, 0.04, 1.0])  # x, y, width, height in axes fraction
    cbar = ax.get_figure().colorbar(im, cax=cax)
    cbar.set_label("Distance (px)", fontsize=12)
    cbar.ax.tick_params(labelsize=12)


    fig.tight_layout()
    plt.show()

def processImage(name, samplename):
  global results_df

  # Read Image File
  image = cv2.imread(name)

  # Perform color deconvolution
  hematoxylin_channel, eosin_channel = perform_color_deconvolution(image)

  # Create masks for Hematoxylin and Eosin channels
  hematoxylin_mask = create_tissue_mask(hematoxylin_channel, mask_threshold)
  eosin_mask = create_tissue_mask(eosin_channel, mask_threshold)


  # Threshold the image to create a binary mask
  _, binary_image = cv2.threshold(hematoxylin_mask, 1, 255, cv2.THRESH_BINARY_INV)
  kernel = np.ones((1,1), np.uint8)

  # Dilate the mask by 1x1 to eliminate small imperfections
  dilated = cv2.dilate(binary_image, kernel, iterations=2)
  _, dilated = cv2.threshold(dilated, 100, 255, cv2.THRESH_BINARY_INV)
  inverted_image = 255 - dilated

  # Find contours
  contours, hierarchy = cv2.findContours(inverted_image, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

  # Create an image to draw the contours on
  contour_image = np.zeros_like(image)

  cv2.drawContours(contour_image, contours, -1, (0, 0, 255), 2)
  hier = hierarchy[0]  # shape (N, 4): [next, prev, first_child, parent]
  outer_idxs = np.where(hier[:, 3] == -1)[0]

  largest_idx = max(outer_idxs, key=lambda i: cv2.contourArea(contours[i]))
  largest_contour = contours[largest_idx]
  max_area = cv2.contourArea(largest_contour)

  hole_idxs = np.where(hier[:, 3] == largest_idx)[0]
  big_hole_idxs = [i for i in hole_idxs if abs(cv2.contourArea(contours[i])) >= (hole_size_percent/100)*max_area]

  # Build a filled mask that keeps holes at least hole_size_percent % of the area of the largest contour (smaller holes are filled)
  largest_contour_mask = np.zeros_like(hematoxylin_mask, dtype=np.uint8)
  cv2.drawContours(largest_contour_mask, [largest_contour], -1, 255, thickness=-1)
  for i in big_hole_idxs:
    cv2.drawContours(largest_contour_mask, contours, i, 0, thickness=-1)

  contour_image = np.zeros_like(image)
  fill_with_holes = np.zeros_like(hematoxylin_mask, dtype=np.uint8)
  cv2.drawContours(fill_with_holes, [largest_contour], -1, 255, thickness=-1)
  for i in big_hole_idxs:
      cv2.drawContours(fill_with_holes, contours, i, 0, thickness=-1)
  contour_image[fill_with_holes == 255] = (255, 255, 255)

  # Create distance transform of largest countour
  dist_transform = cv2.distanceTransform(largest_contour_mask, cv2.DIST_L2, 5)
  dist = distance_transform_edt(largest_contour_mask, return_indices=False, return_distances=True)


  # Skeletonize largest countour and apply skeleton pruning algorithm
  skeleton_mask = largest_contour_mask.copy()
  skeleton_mask = cv2.erode(skeleton_mask, np.ones((3,3), np.uint8), cv2.BORDER_CONSTANT, iterations=2)
  skeleton = skeletonize(skeleton_mask)
  skeleton = skel_pruning_DSE(skeleton, dist, dse_pruning_threshold)

  # Apply distance transform values to skeleton pixels
  skeleton_pixels = skeleton > 0
  skeleton_distances = dist_transform[skeleton_pixels]
  skeleton_distances = skeleton_distances[skeleton_distances > min_thickness]

  # Compute the mean distance transform value for the skeleton pixels
  mean_distance = 2*np.mean(skeleton_distances)

  length = len(skeleton_distances)

  mean_distance = PixelstoUm(mean_distance)
  print(f"Average Width: {mean_distance} microns")


  # Create data frame with results
  skeleton_distances_df = pd.DataFrame(skeleton_distances, columns=['Skeleton Distances (Pixels)'])
  skeleton_distances_df.to_csv(f'{samplename}skeleton_distances.csv', index=False)
  results_df = pd.concat([results_df, skeleton_distances_df], ignore_index=True)

  show_periosteum_panels(image, inverted_image, contour_image, skeleton, dist_transform, largest_contour_mask)

  return mean_distance, max_area, length, length*mean_distance

# Reprocess image if the largest contour is not the periosteum
def ReprocessImage(name, maxcontour, samplename):
  global results_df

  image = cv2.imread(name)
  if image is None:
    raise FileNotFoundError(f"Could not read image: {name}")

  # Perform color deconvolution
  hematoxylin_channel, eosin_channel = perform_color_deconvolution(image)

  # Create masks for Hematoxylin and Eosin channels
  hematoxylin_mask = create_tissue_mask(hematoxylin_channel, mask_threshold)
  eosin_mask = create_tissue_mask(eosin_channel, mask_threshold)

  # Threshold the image to create a binary mask  (keep your original thresholds)
  _, binary_image = cv2.threshold(hematoxylin_mask, 200, 255, cv2.THRESH_BINARY_INV)
  kernel = np.ones((1,1), np.uint8)
  dilated = cv2.dilate(binary_image, kernel, iterations=2)
  _, dilated = cv2.threshold(dilated, 200, 255, cv2.THRESH_BINARY_INV)
  inverted_image = 255 - dilated

  # Find contours (external only, as before)
  contours, _ = cv2.findContours(inverted_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
  if len(contours) == 0:
    print(f"[WARN] No contours found for {name}")
    contour_image = np.zeros_like(image)
    dist_transform = np.zeros_like(hematoxylin_mask, dtype=np.float32)
    skeleton = np.zeros_like(hematoxylin_mask, dtype=bool)
    show_periosteum_panels(image, inverted_image, contour_image, skeleton, dist_transform)
    return np.nan, 0, 0, np.nan

  # Pick largest contour under the area cap
  best_contour = None
  best_area = 0.0
  maxcontour = float(maxcontour)

  for c in contours:
    a = float(cv2.contourArea(c))
    if a > best_area and a < maxcontour:
      best_area = a
      best_contour = c


  # Build mask for selected contour
  largest_contour_mask = np.zeros_like(hematoxylin_mask, dtype=np.uint8)
  cv2.drawContours(largest_contour_mask, [best_contour], -1, 255, -1)

  # Build a contour_image for visualization (white fill)
  contour_image = np.zeros_like(image)
  contour_image[largest_contour_mask == 255] = (255, 255, 255)

  # Apply distance transform
  dist_transform = cv2.distanceTransform(largest_contour_mask, cv2.DIST_L2, 5)
  dist = distance_transform_edt(largest_contour_mask, return_indices=False, return_distances=True)

  # Skeletonize + prune
  skeleton_mask = cv2.erode(
      largest_contour_mask,
      np.ones((3,3), np.uint8),
      borderType=cv2.BORDER_CONSTANT,
      iterations= 2 )
  skeleton = skeletonize(skeleton_mask)

  skeleton = skel_pruning_DSE(skeleton, dist, dse_pruning_threshold)

  # Apply distance transform
  skeleton_pixels = skeleton > 0
  skeleton_distances = dist_transform[skeleton_pixels].astype(float)
  skeleton_distances = skeleton_distances[np.isfinite(skeleton_distances)]
  skeleton_distances = skeleton_distances[skeleton_distances > min_thickness]

  # Width from distances
  mean_distance = 2*np.mean(skeleton_distances)
  length = int(len(skeleton_distances))

  mean_distance = PixelstoUm(mean_distance)
  print(f"Average Width: {mean_distance} microns")
  stem = os.path.splitext(os.path.basename(name))[0].replace(" ", "_")
  csv_name = f"{samplename}_{stem}_skeleton_distances.csv"

  skeleton_distances_df = pd.DataFrame(
      skeleton_distances,
      columns=['Skeleton Distances (Pixels)']
  )
  skeleton_distances_df.to_csv(csv_name, index=False)

  # Keep aggregated df consistent with what was saved
  results_df = pd.concat([results_df, skeleton_distances_df], ignore_index=True)

  show_periosteum_panels(image, inverted_image, contour_image, skeleton, dist_transform, largest_contour_mask)

  return mean_distance, best_area, length, mean_distance*length

#Process Batch of Images found on Google Drive
def processBatch(samplename, number):
  df = pd.DataFrame()
  for i in range(number):
    sample = f"{samplename}{i+1}"
    print(f"{samplename}{i+1}.jpg")
    df[i]= processImage(f"/content/drive/MyDrive/Periosteum Quantification Images/{sample}.jpg", sample)


  repeat = 0
  while repeat != "No":
    repeat = input('Please enter image number to repeat, or enter "No" when done: ')
    if repeat == "No":
      break
    repeat = int(repeat)
    for i in range(number):
      if i+1 == repeat:
        sample = f"{samplename}{i+1}"
        print(f"{samplename}{i+1}.jpg")
        df[i]= ReprocessImage(f"/content/drive/MyDrive/Periosteum Quantification Images/{sample}.jpg", df.loc[1, repeat-1], sample)
  print(df)
  average = df.iloc[3].sum()/(df.iloc[2].sum())
  results_df.to_csv("results.csv", index=False)
  return average

## 4. Parameters

Set these for your own images. The functions read them when the pipeline runs, so after changing a value you only need to rerun this cell. The comment after each value is what was used for the manuscript.

In [ ]:
# Stain vectors for color deconvolution, one stain per column
# (adjust as necessary for your stain; the third column stays as zeros for two stains)
stain_matrix = np.array([
    [0.123, 0.4,   0.0],
    [0.509, 0.914, 0.0],
    [0.655, 0.43,  0.0],
])

# Intensity threshold (0-255) applied to the stain channel to make the binary mask
mask_threshold = 155  

# Holes inside the periosteum smaller than this percentage of its contour area are filled in, holes this size or larger are kept as holes
hole_size_percent = 1 

# DSE pruning threshold in pixels: a skeleton branch is removed if deleting it loses fewer than this many pixels of the reconstructed shape
dse_pruning_threshold = 2500  

# Minimum thickness filter: skeleton points with a distance transform value at or below this are left out of the measurement. 
min_thickness = 4.0 

# Scale factor: pixels per micron for your microscope and magnification
pixels_per_um = 3.966  

## 5. Run on your images

Images go in `My Drive/Periosteum Quantification Images/` and are named with a shared prefix and a number starting at 1 (for example `WT1.jpg`, `WT2.jpg`). Change the prefix and image count below.

After the batch finishes you will be asked for an image number to redo. Enter `No` when you are done.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

average_width = processBatch("WT", 3)  # (image name prefix, number of images)
print(f"Average Thickness: {average_width} microns")